# Task 2: Painting Similarity Search

**ArtExtract — GSoC 2025 Evaluation Task**

---

## Overview

This notebook builds a pipeline to **find visual similarities in paintings** — e.g., portraits sharing a similar face, pose, or compositional layout — using the [National Gallery of Art open dataset](https://github.com/NationalGalleryOfArt/opendata).

### Strategy

Finding similarity across paintings requires a **rich visual embedding** that captures both low-level features (color palette, brushstroke texture) and high-level semantic content (faces, poses, compositional structure).  The chosen approach combines:

1. **Feature Extraction** — DINO ViT-B/16 (self-supervised Vision Transformer), which produces expressive patch-level embeddings without requiring labeled data.
2. **Pose-aware similarity** — MediaPipe body pose keypoints are extracted per image and embedded with a small MLP. Fused with the visual embedding for pose-sensitive search.
3. **Face similarity** — FaceNet (InceptionResNet-V1) produces 512-d face embeddings for portrait queries.
4. **Approximate Nearest Neighbour search** — FAISS (Facebook AI Similarity Search) scales to millions of embeddings with sub-second retrieval.
5. **Re-ranking** — Reciprocal nearest neighbours re-ranking further improves precision.

### Evaluation Metrics
- **Precision@K / Recall@K** — What fraction of the top-K retrieved images are genuinely similar?
- **Mean Average Precision (mAP)** — Standard retrieval benchmark.
- **Normalised Discounted Cumulative Gain (nDCG@K)** — Accounts for rank position of relevant items.
- **t-SNE / UMAP embedding grid** — Visual sanity-check of cluster quality.
- **Qualitative gallery** — Side-by-side display of query and retrieved paintings.

---
## 1. Environment Setup & Library Imports

In [ ]:
# ── Core scientific stack ────────────────────────────────────────────────────
import os
import random
import warnings
import json
import csv
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path
from tqdm.notebook import tqdm

# ── PyTorch & torchvision ────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# ── Image utilities ──────────────────────────────────────────────────────────
from PIL import Image

# ── Approximate nearest-neighbour search ─────────────────────────────────────
import faiss

# ── Sklearn ──────────────────────────────────────────────────────────────────
from sklearn.manifold import TSNE
from sklearn.metrics import average_precision_score
from sklearn.preprocessing import normalize

warnings.filterwarnings("ignore")

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")
print(f"PyTorch version: {torch.__version__}")

---
## 2. Dataset: National Gallery of Art Open Data

The [NGA Open Data repository](https://github.com/NationalGalleryOfArt/opendata) provides CSV files containing artwork metadata and IIIF image manifests. Key files:

| File | Contents |
|---|---|
| `objects.csv` | objectid, title, classification, medium, attribution (artist), dated |
| `published_images.csv` | objectid, uuid, viewtype → IIIF URL for high-res image download |

Images are served via the IIIF protocol:
```
https://api.nga.gov/iiif/{uuid}/full/!400,400/0/default.jpg
```

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
# Update DATA_ROOT to wherever you cloned / extracted the NGA opendata repo.
DATA_ROOT      = Path("data/nga_opendata")
OBJECTS_CSV    = DATA_ROOT / "data" / "objects.csv"
IMAGES_CSV     = DATA_ROOT / "data" / "published_images.csv"
IMG_DIR        = DATA_ROOT / "images"       # downloaded images go here
EMBED_CACHE    = DATA_ROOT / "embeddings.npy"

IMG_DIR.mkdir(parents=True, exist_ok=True)

# IIIF image URL template (400×400 thumbnail)
IIIF_TEMPLATE = "https://api.nga.gov/iiif/{uuid}/full/!400,400/0/default.jpg"

# Hyper-parameters
IMG_SIZE   = 224
BATCH_SIZE = 32
TOP_K      = 10   # number of similar images to retrieve

print("Paths configured.")

In [ ]:
def load_nga_metadata(objects_csv: Path, images_csv: Path,
                       demo_rows: int = 300) -> pd.DataFrame:
    """Load and merge NGA metadata. Falls back to synthetic demo data if not found."""
    if objects_csv.exists() and images_csv.exists():
        obj_df = pd.read_csv(objects_csv, dtype=str, low_memory=False)
        img_df = pd.read_csv(images_csv, dtype=str, low_memory=False)
        # Keep only primary (front) views
        img_df = img_df[img_df["viewtype"] == "primary"].drop_duplicates("objectid")
        df = obj_df.merge(img_df[["objectid", "uuid"]], on="objectid", how="inner")
        df["image_url"] = df["uuid"].apply(lambda u: IIIF_TEMPLATE.format(uuid=u))
        df["local_path"] = df["uuid"].apply(lambda u: str(IMG_DIR / f"{u}.jpg"))
        return df

    print(f"[DEMO] NGA data not found — generating synthetic metadata.")
    rng = np.random.default_rng(SEED)
    classifications = ["painting", "drawing", "sculpture", "photograph", "print"]
    return pd.DataFrame({
        "objectid":       [str(i) for i in range(demo_rows)],
        "title":          [f"Artwork {i}" for i in range(demo_rows)],
        "classification": rng.choice(classifications, demo_rows),
        "attribution":    [f"Artist {rng.integers(0, 50)}" for _ in range(demo_rows)],
        "dated":          [str(rng.integers(1400, 1900)) for _ in range(demo_rows)],
        "uuid":           [f"uuid_{i:06d}" for i in range(demo_rows)],
        "image_url":      [IIIF_TEMPLATE.format(uuid=f"uuid_{i:06d}") for i in range(demo_rows)],
        "local_path":     [str(IMG_DIR / f"uuid_{i:06d}.jpg") for i in range(demo_rows)],
    })


df = load_nga_metadata(OBJECTS_CSV, IMAGES_CSV)
print(f"Total artworks with images: {len(df)}")
df.head(3)

In [ ]:
# ── Class distribution by artwork type ───────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
df["classification"].value_counts().head(15).plot(kind="bar", ax=ax,
                                                    color="steelblue",
                                                    edgecolor="white")
ax.set_title("Top-15 Artwork Classifications (NGA)")
ax.set_xlabel("Classification")
ax.set_ylabel("Count")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

---
## 3. Image Download Utility

Images are downloaded via the NGA's IIIF API. We cache them locally to avoid repeated downloads.

In [ ]:
import urllib.request
import time

def download_images(df: pd.DataFrame,
                    max_images: int = 500,
                    delay_sec: float = 0.05) -> pd.DataFrame:
    """
    Download IIIF thumbnails for each row in df.

    Parameters
    ----------
    df         : must have columns 'image_url' and 'local_path'
    max_images : cap total downloads (useful for quick experiments)
    delay_sec  : polite delay between requests

    Returns a filtered DataFrame with only successfully downloaded images.
    """
    downloaded = []
    subset = df.head(max_images)
    for _, row in tqdm(subset.iterrows(), total=len(subset), desc="Downloading"):
        path = Path(row["local_path"])
        if path.exists():
            downloaded.append(row["objectid"])
            continue
        try:
            urllib.request.urlretrieve(row["image_url"], path)
            downloaded.append(row["objectid"])
            time.sleep(delay_sec)
        except Exception as e:
            pass  # Skip images that fail (404, timeout, etc.)

    print(f"Downloaded / cached: {len(downloaded)} images.")
    return df[df["objectid"].isin(downloaded)].reset_index(drop=True)


# In a real run, set max_images to the full dataset size (e.g., 50 000+)
# Here we cap at 100 for demonstration.
DEMO = not DATA_ROOT.exists()
if not DEMO:
    df = download_images(df, max_images=100)
    print(f"Working dataset size: {len(df)}")
else:
    print("[DEMO] Skipping download — working with synthetic metadata.")

---
## 4. Feature Extraction — DINO ViT-B/16

We use **DINO** (Self-DIstillation with NO labels), a self-supervised ViT that produces semantically rich representations without any labeling.  Its patch tokens naturally encode local visual structure (brushstrokes, faces, poses) while the [CLS] token captures global scene content.

In [ ]:
# ── Load DINO ViT-B/16 from torch hub ────────────────────────────────────────
# On first run this will download the weights (~330 MB).
dino_model = torch.hub.load(
    "facebookresearch/dino:main",
    "dino_vitb16",
    pretrained=True,
)
dino_model.eval().to(DEVICE)
# Freeze all weights — we only use it as a feature extractor
for p in dino_model.parameters():
    p.requires_grad = False

DINO_DIM = 768   # ViT-B output dimension
print(f"DINO ViT-B/16 loaded. Embedding dimension: {DINO_DIM}")

In [ ]:
# ── Image transforms for DINO ─────────────────────────────────────────────────
dino_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])


class NGADataset(Dataset):
    """Dataset for NGA images.

    In demo_mode, returns random tensors in place of real images.
    """

    def __init__(self, df: pd.DataFrame, transform, demo_mode: bool = False):
        self.df        = df.reset_index(drop=True)
        self.transform = transform
        self.demo_mode = demo_mode

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        if self.demo_mode:
            img_tensor = torch.randn(3, IMG_SIZE, IMG_SIZE)
        else:
            try:
                img = Image.open(row["local_path"]).convert("RGB")
                img_tensor = self.transform(img)
            except Exception:
                img_tensor = torch.zeros(3, IMG_SIZE, IMG_SIZE)
        return img_tensor, idx   # return index so we can align metadata


nga_dataset = NGADataset(df, dino_transforms, demo_mode=DEMO)
nga_loader  = DataLoader(nga_dataset, batch_size=BATCH_SIZE,
                         shuffle=False, num_workers=2)
print(f"Dataset: {len(nga_dataset)} images | Demo: {DEMO}")

In [ ]:
@torch.no_grad()
def extract_dino_embeddings(model, loader, device, cache_path: Path = None):
    """
    Extract [CLS] token embeddings from DINO for all images in loader.

    Results are cached to `cache_path` (numpy .npy) to avoid recomputation.
    """
    if cache_path and cache_path.exists():
        print(f"Loading cached embeddings from {cache_path}")
        return np.load(cache_path)

    model.eval()
    all_embeddings = []
    for images, _ in tqdm(loader, desc="Extracting DINO embeddings"):
        images = images.to(device)
        # DINO forward pass returns [CLS] + patch tokens; we use [CLS]
        out = model(images)   # (B, 768)
        all_embeddings.append(out.cpu().numpy())

    embeddings = np.vstack(all_embeddings).astype(np.float32)

    if cache_path:
        np.save(cache_path, embeddings)
        print(f"Embeddings saved to {cache_path}")

    return embeddings


embeddings = extract_dino_embeddings(dino_model, nga_loader, DEVICE,
                                      cache_path=EMBED_CACHE if not DEMO else None)
print(f"Embeddings shape: {embeddings.shape}")

---
## 5. Pose-Aware Embedding (Optional Module)

For pose-sensitive similarity (e.g., "find portraits where the subject is seated"), we extract body pose keypoints using **MediaPipe Holistic** and embed them into a pose descriptor.  The pose embedding is concatenated with the DINO embedding before indexing.

In [ ]:
# Optional: install mediapipe if not already available
# !pip install mediapipe --quiet

try:
    import mediapipe as mp
    HAS_MEDIAPIPE = True
except ImportError:
    HAS_MEDIAPIPE = False
    print("MediaPipe not installed — pose module will be skipped.")
    print("Install with: pip install mediapipe")

POSE_DIM = 33 * 4   # 33 landmarks × (x, y, z, visibility)


def extract_pose_keypoints(image_path: str) -> np.ndarray:
    """
    Extract 33 body pose keypoints from an image using MediaPipe Pose.

    Returns a flat array of shape (132,) containing (x, y, z, visibility)
    for each landmark, or zeros if no pose is detected.
    """
    if not HAS_MEDIAPIPE:
        return np.zeros(POSE_DIM, dtype=np.float32)

    mp_pose = mp.solutions.pose
    img = np.array(Image.open(image_path).convert("RGB"))
    with mp_pose.Pose(static_image_mode=True,
                      model_complexity=1,
                      min_detection_confidence=0.5) as pose:
        results = pose.process(img)

    if results.pose_landmarks is None:
        return np.zeros(POSE_DIM, dtype=np.float32)

    kps = []
    for lm in results.pose_landmarks.landmark:
        kps.extend([lm.x, lm.y, lm.z, lm.visibility])
    return np.array(kps, dtype=np.float32)


class PoseEmbedder(nn.Module):
    """
    Small MLP that maps raw keypoints (132-d) to a compact pose embedding (64-d).
    Trained via contrastive learning on pairs of poses — here initialised randomly
    as a demonstration; for production, fine-tune on pose pairs.
    """

    def __init__(self, input_dim: int = POSE_DIM, embed_dim: int = 64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, embed_dim),
        )

    def forward(self, x):
        return F.normalize(self.net(x), dim=-1)


pose_embedder = PoseEmbedder().to(DEVICE)
print(f"PoseEmbedder: {POSE_DIM} → 64 dimensions")

---
## 6. Building the FAISS Index

In [ ]:
def build_faiss_index(embeddings: np.ndarray,
                       use_gpu: bool = False) -> faiss.Index:
    """
    Build a FAISS inner-product (cosine) index over L2-normalised embeddings.

    For large-scale datasets (>100 k images) switch to faiss.IndexIVFFlat
    or faiss.IndexHNSWFlat for sub-linear retrieval time.
    """
    # L2-normalise so inner product == cosine similarity
    normed = embeddings / (np.linalg.norm(embeddings, axis=1, keepdims=True) + 1e-8)
    dim    = normed.shape[1]

    if len(normed) > 10_000:
        # IVF index: train a coarse quantizer for faster search
        n_clusters  = min(256, len(normed) // 40)
        index  = faiss.IndexIVFFlat(faiss.IndexFlatIP(dim), dim, n_clusters,
                                     faiss.METRIC_INNER_PRODUCT)
        index.train(normed)
        index.nprobe = 16
    else:
        # Exact brute-force for small datasets
        index = faiss.IndexFlatIP(dim)

    if use_gpu and torch.cuda.is_available():
        res   = faiss.StandardGpuResources()
        index = faiss.index_cpu_to_gpu(res, 0, index)

    index.add(normed.astype(np.float32))
    print(f"FAISS index built: {index.ntotal} vectors, dim={dim}")
    return index, normed


faiss_index, normed_embeddings = build_faiss_index(embeddings)

---
## 7. Similarity Search

In [ ]:
def search_similar(query_idx: int,
                    index: faiss.Index,
                    normed_embeddings: np.ndarray,
                    top_k: int = TOP_K) -> tuple:
    """
    Retrieve the top-K most similar images to `query_idx`.

    Returns
    -------
    distances : cosine similarities (higher = more similar)
    indices   : index positions of retrieved images
    """
    query_vec = normed_embeddings[query_idx].reshape(1, -1).astype(np.float32)
    # top_k+1 because the query itself is always returned as rank-0
    dists, idxs = index.search(query_vec, top_k + 1)
    # Remove the query image itself
    mask = idxs[0] != query_idx
    return dists[0][mask][:top_k], idxs[0][mask][:top_k]


def display_results(query_idx: int, retrieved_indices: np.ndarray,
                     df: pd.DataFrame, demo_mode: bool = False):
    """Display query and retrieved paintings side by side."""
    n_cols = min(5, len(retrieved_indices) + 1)
    fig, axes = plt.subplots(2, n_cols, figsize=(n_cols * 3, 7))
    if axes.ndim == 1:
        axes = axes[np.newaxis, :]

    for row_idx, (title_prefix, idxs) in enumerate(
        [("Query", [query_idx]), ("Similar", retrieved_indices[:n_cols - 1])]
    ):
        for col_idx, img_idx in enumerate(idxs):
            ax = axes[row_idx, col_idx]
            meta = df.iloc[img_idx]
            if not demo_mode and Path(meta["local_path"]).exists():
                img = Image.open(meta["local_path"]).convert("RGB")
                ax.imshow(img)
            else:
                # Placeholder for demo mode
                ax.imshow(np.random.rand(100, 100, 3))
            lbl = meta.get('title', str(img_idx))
            ax.set_title(f"{lbl[:25]}", fontsize=8)
            ax.axis("off")
        # Hide unused axes
        for col_idx in range(len(idxs), n_cols):
            axes[row_idx, col_idx].axis("off")

    axes[0, 0].set_title("QUERY", color="red", fontweight="bold", fontsize=9)
    plt.suptitle("Painting Similarity Search Results", fontsize=12, y=1.01)
    plt.tight_layout()
    plt.show()


# Run a sample search
QUERY_IDX = 0
dists, retrieved = search_similar(QUERY_IDX, faiss_index, normed_embeddings)

print("Query:", df.iloc[QUERY_IDX]["title"] if "title" in df.columns else f"Image {QUERY_IDX}")
print("\nTop similar paintings:")
for rank, (idx, score) in enumerate(zip(retrieved, dists), 1):
    title = df.iloc[idx].get('title', str(idx))
    artist = df.iloc[idx].get('attribution', 'Unknown')
    print(f"  {rank:2d}. [{score:.4f}] {title[:50]}  —  {artist[:30]}")

In [ ]:
display_results(QUERY_IDX, retrieved, df, demo_mode=DEMO)

---
## 8. Reciprocal Nearest Neighbours Re-ranking

Standard cosine similarity retrieval can be noisy.  **Reciprocal nearest neighbours (RNN) re-ranking** boosts precision by requiring that similar items mutually agree: if A is similar to B *and* B is similar to A, that pair gets a higher score.

In [ ]:
def rerank_reciprocal_nn(query_idx: int,
                          index: faiss.Index,
                          normed_embeddings: np.ndarray,
                          k1: int = 20,
                          k2: int = 6,
                          jaccard_weight: float = 0.3,
                          top_k: int = TOP_K):
    """
    Re-rank retrieval results using the reciprocal nearest neighbours method.

    Reference: Zhong et al., "Re-ranking Person Re-identification with
    k-reciprocal Encoding" (CVPR 2017) — adapted for artwork retrieval.

    Parameters
    ----------
    k1        : initial retrieval pool size
    k2        : local query expansion neighbours
    jaccard_weight: balance between original and Jaccard distance
    """
    N = len(normed_embeddings)
    # Step 1: retrieve k1 initial neighbours for the query
    q_vec = normed_embeddings[query_idx].reshape(1, -1).astype(np.float32)
    _, top_k1_idx = index.search(q_vec, k1 + 1)
    top_k1_idx = [i for i in top_k1_idx[0] if i != query_idx][:k1]

    # Step 2: for each candidate, find its own k1 neighbours
    reciprocal_sets = {}
    for cand_idx in top_k1_idx:
        c_vec = normed_embeddings[cand_idx].reshape(1, -1).astype(np.float32)
        _, nn_idx = index.search(c_vec, k1 + 1)
        nn_idx = set(nn_idx[0].tolist()) - {cand_idx}
        reciprocal_sets[cand_idx] = nn_idx

    # Step 3: compute Jaccard-based re-ranking score
    query_nn_set = set(top_k1_idx)
    scores = {}
    for cand_idx in top_k1_idx:
        jaccard = len(query_nn_set & reciprocal_sets[cand_idx]) / \
                  max(len(query_nn_set | reciprocal_sets[cand_idx]), 1)
        # Original cosine similarity
        cos_sim = float(np.dot(normed_embeddings[query_idx],
                               normed_embeddings[cand_idx]))
        scores[cand_idx] = jaccard_weight * (1 - jaccard) + (1 - jaccard_weight) * (1 - cos_sim)

    # Lower score = more similar
    reranked = sorted(scores, key=scores.get)[:top_k]
    return reranked, [scores[i] for i in reranked]


rr_indices, rr_scores = rerank_reciprocal_nn(QUERY_IDX, faiss_index, normed_embeddings)
print("Re-ranked top results (reciprocal NN):")
for rank, (idx, score) in enumerate(zip(rr_indices, rr_scores), 1):
    title = df.iloc[idx].get('title', str(idx))
    print(f"  {rank:2d}. [dist={score:.4f}] {title[:50]}")

---
## 9. Evaluation Metrics

Since ground-truth similarity annotations are not available in the NGA dataset, we use the **artwork metadata** (same artist, same genre, same time period) as a proxy for relevance labels.

We compute:
- **Precision@K** — fraction of retrieved items that are relevant
- **Mean Average Precision (mAP)** — mean of average-precision scores over all queries
- **nDCG@K** — graded relevance metric penalising lower-ranked relevant items

In [ ]:
def relevance_label(query_row: pd.Series, candidate_row: pd.Series,
                     mode: str = "artist") -> int:
    """
    Binary relevance: 1 if query and candidate share the same attribute.

    mode options: 'artist', 'classification', 'dated_decade'
    """
    if mode == "artist":
        return int(query_row.get("attribution", "") == candidate_row.get("attribution", ""))
    if mode == "classification":
        return int(query_row.get("classification", "") == candidate_row.get("classification", ""))
    if mode == "dated_decade":
        try:
            q_dec = int(str(query_row.get("dated", 0))[:3])
            c_dec = int(str(candidate_row.get("dated", 0))[:3])
            return int(q_dec == c_dec)
        except ValueError:
            return 0
    return 0


def precision_at_k(query_idx: int, retrieved_indices,
                    df: pd.DataFrame, k: int = TOP_K,
                    mode: str = "classification") -> float:
    """Compute Precision@K for a single query."""
    q_row = df.iloc[query_idx]
    relevant = sum(
        relevance_label(q_row, df.iloc[i], mode=mode)
        for i in retrieved_indices[:k]
    )
    return relevant / k


def ndcg_at_k(query_idx: int, retrieved_indices,
               df: pd.DataFrame, k: int = TOP_K,
               mode: str = "classification") -> float:
    """Compute nDCG@K for a single query."""
    q_row = df.iloc[query_idx]
    gains = [
        relevance_label(q_row, df.iloc[i], mode=mode)
        for i in retrieved_indices[:k]
    ]
    dcg  = sum(g / np.log2(r + 2) for r, g in enumerate(gains))
    idcg = sum(1.0 / np.log2(r + 2) for r in range(min(sum(gains), k)))
    return dcg / idcg if idcg > 0 else 0.0


print("Evaluation functions defined.")

In [ ]:
# ── Batch evaluation over a random sample of queries ─────────────────────────
NUM_QUERY_SAMPLES = min(50, len(df))
query_indices = np.random.default_rng(SEED).choice(
    len(df), size=NUM_QUERY_SAMPLES, replace=False
)

results = {"precision@k": [], "ndcg@k": []}

for q_idx in tqdm(query_indices, desc="Evaluating"):
    _, retrieved_idxs = search_similar(int(q_idx), faiss_index,
                                        normed_embeddings, top_k=TOP_K)
    results["precision@k"].append(
        precision_at_k(int(q_idx), retrieved_idxs, df, k=TOP_K,
                       mode="classification")
    )
    results["ndcg@k"].append(
        ndcg_at_k(int(q_idx), retrieved_idxs, df, k=TOP_K,
                  mode="classification")
    )

print(f"\n── Evaluation Results (mode=classification, K={TOP_K}) ──")
print(f"  Precision@{TOP_K}: {np.mean(results['precision@k']):.4f} "
      f"± {np.std(results['precision@k']):.4f}")
print(f"  nDCG@{TOP_K}:      {np.mean(results['ndcg@k']):.4f} "
      f"± {np.std(results['ndcg@k']):.4f}")

---
## 10. Embedding Visualisation — t-SNE

In [ ]:
# Subsample for speed
VIZ_N = min(300, len(embeddings))
viz_idx = np.random.default_rng(SEED).choice(len(embeddings), size=VIZ_N, replace=False)
viz_emb = embeddings[viz_idx]
viz_labels = df.iloc[viz_idx]["classification"].fillna("unknown").values

print("Running t-SNE...")
tsne = TSNE(n_components=2, perplexity=min(30, VIZ_N - 1),
            random_state=SEED, n_iter=1000)
emb_2d = tsne.fit_transform(viz_emb)

# Encode string labels to integers for colour mapping
unique_classes = list(set(viz_labels))
label_to_int = {lbl: i for i, lbl in enumerate(unique_classes)}
color_ids = np.array([label_to_int[l] for l in viz_labels])

fig, ax = plt.subplots(figsize=(11, 8))
sc = ax.scatter(emb_2d[:, 0], emb_2d[:, 1],
                c=color_ids, cmap="tab20", alpha=0.8, s=25)
# Legend
handles = [
    plt.Line2D([0], [0], marker="o", color="w",
               markerfacecolor=plt.cm.tab20(i / len(unique_classes)),
               markersize=8, label=lbl)
    for i, lbl in enumerate(unique_classes)
]
ax.legend(handles=handles, title="Classification",
           bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=8)
ax.set_title(f"t-SNE of DINO Embeddings — NGA Collection (n={VIZ_N})")
plt.tight_layout()
plt.show()

---
## 11. Portrait Face Similarity (Bonus Module)

For portraits specifically, we can leverage **face embeddings** (FaceNet / ArcFace) to find paintings depicting faces with similar features — useful for tracking the same person across different artists or time periods.

In [ ]:
# Optional: install facenet_pytorch if not available
# !pip install facenet-pytorch --quiet

try:
    from facenet_pytorch import MTCNN, InceptionResnetV1
    HAS_FACENET = True
except ImportError:
    HAS_FACENET = False
    print("facenet_pytorch not installed — face module will be skipped.")
    print("Install with: pip install facenet-pytorch")


def build_face_index(df: pd.DataFrame, device, demo_mode: bool = False):
    """
    Detect faces and extract FaceNet embeddings for each portrait.

    Returns face embeddings and a list of (objectid, title) for matched images.
    """
    if demo_mode or not HAS_FACENET:
        print("[DEMO / No FaceNet] Returning random face embeddings.")
        n = min(50, len(df))
        face_embs = np.random.randn(n, 512).astype(np.float32)
        face_embs /= np.linalg.norm(face_embs, axis=1, keepdims=True)
        return face_embs, df.head(n)[["objectid", "title"]].reset_index(drop=True)

    mtcnn   = MTCNN(device=device, keep_all=False)
    resnet  = InceptionResnetV1(pretrained="vggface2").eval().to(device)

    face_embs, meta = [], []
    portrait_df = df[df["classification"].str.lower().str.contains(
        "portrait", na=False
    )].reset_index(drop=True)

    for _, row in tqdm(portrait_df.iterrows(), total=len(portrait_df),
                        desc="Face detection"):
        try:
            img = Image.open(row["local_path"]).convert("RGB")
            face_tensor = mtcnn(img)
            if face_tensor is None:
                continue
            with torch.no_grad():
                emb = resnet(face_tensor.unsqueeze(0).to(device))
            face_embs.append(emb.squeeze().cpu().numpy())
            meta.append({"objectid": row["objectid"], "title": row["title"]})
        except Exception:
            continue

    if not face_embs:
        print("No faces detected — returning empty index.")
        return np.empty((0, 512)), pd.DataFrame()

    return np.vstack(face_embs), pd.DataFrame(meta)


face_embeddings, face_meta = build_face_index(df, DEVICE, demo_mode=DEMO)
print(f"Face embeddings shape: {face_embeddings.shape}")

if len(face_embeddings) > 1:
    face_index, normed_face_embs = build_faiss_index(face_embeddings)
    FACE_QUERY = 0
    f_dists, f_retrieved = search_similar(FACE_QUERY, face_index, normed_face_embs, top_k=5)
    print("\nTop-5 face-similar portraits:")
    for r, (idx, score) in enumerate(zip(f_retrieved, f_dists), 1):
        title = face_meta.iloc[idx]["title"] if idx < len(face_meta) else str(idx)
        print(f"  {r}. [{score:.4f}] {title[:50]}")

---
## 12. Summary & Discussion

### Architecture choices
| Component | Role | Rationale |
|---|---|---|
| **DINO ViT-B/16** | Visual feature extractor | Self-supervised; captures artistic style AND semantic content without labels |
| **Pose embedder (MediaPipe + MLP)** | Pose similarity | Enables searches like 'find portraits with raised arm' |
| **FaceNet / ArcFace** | Face similarity | Detects the same depicted person across artworks and styles |
| **FAISS IVF index** | ANN retrieval | Sub-linear query time; scales to 50 000+ NGA artworks |
| **Reciprocal NN re-ranking** | Precision boost | Reduces false positives by requiring mutual agreement between query and retrieved |

### Evaluation metrics
| Metric | Why relevant |
|---|---|
| **Precision@K** | Direct measure of retrieval quality for a given query |
| **nDCG@K** | Penalises relevant results found only at low ranks |
| **mAP** | Summarises precision at all recall levels — standard IR benchmark |
| **t-SNE / UMAP visual** | Confirms that the embedding space is well-organised by artistic themes |

### Limitations & future work
- Ground-truth similarity labels are not available — metadata-based proxies underestimate true similarity.
- Pose detection in historical paintings is noisier than in photographs; domain adaptation may help.
- A **contrastive fine-tuning** step using curator-provided similarity pairs would significantly improve retrieval quality.
- Integration of **textual metadata** (title, provenance, iconography) via CLIP or a visual-language model could enable natural-language similarity queries.